# Hands-on Task: Can the Cheap Model Handle It?

Runs your RAG assistant's real context+question against two model sizes, recording cost, latency, and a quality judgment for each.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
# Prices are illustrative ($ per 1M tokens) - check the provider's current pricing page for real numbers
PRICING = {"gpt-4o-mini": {"input": 0.15, "output": 0.60}, "gpt-4o": {"input": 2.50, "output": 10.00}}

In [ ]:
import time
def run_and_measure(model, messages):
    start = time.time()
    response = client.chat.completions.create(model=model, messages=messages)
    elapsed = time.time() - start
    usage = response.usage
    cost = (usage.prompt_tokens * PRICING[model]["input"] + usage.completion_tokens * PRICING[model]["output"]) / 1_000_000
    return {"model": model, "seconds": elapsed, "cost": cost, "answer": response.choices[0].message.content}

In [ ]:
context = ("Ojasa Mirai offers a 30-day money-back guarantee for its AI Engineer's course "
           "-- no questions asked. Refund requests must be submitted within 30 days of enrollment.")
question = "What's Ojasa Mirai's refund policy for its courses?"
messages = [{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer using only the context above."}]

In [ ]:
results = [run_and_measure(m, messages) for m in ["gpt-4o-mini", "gpt-4o"]]
for r in results:
    print(r["model"], f"{r['seconds']:.2f}s", f"${r['cost']:.6f}")
    print("  ", r["answer"])

## How to Extend
Swap in a different context+question pair from your own Module 4 pipeline. Add a third model to the comparison and see how the trade-off shifts.

## Done When
You have real cost and latency numbers for both models on the same student question, plus your own one-line verdict on which one your assistant should actually run on.